In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import os

# Configure the plotting/data routines
import pandas as pd

# Import CRUD module
from CRUD_Python_Module import AnimalShelter

# JupyterDash proxy config
JupyterDash.infer_jupyter_proxy_config()


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "2941064"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Load all records initially
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId column because Dash DataTable cannot display it properly
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'

if os.path.exists(image_filename):
    encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()
    logo = html.Img(
        src='data:image/png;base64,{}'.format(encoded_image),
        style={'height': '120px'}
    )
else:
    logo = html.Div("Grazioso Salvare Logo")


app.layout = html.Div([
    html.Center([
        logo,
        html.H1('Grazioso Salvare Rescue Animal Dashboard'),
        html.H3('Austin Schneller - CS-340 Project Two')
    ]),

    html.Hr(),

    html.Div([
        html.Label("Select Rescue Type:", style={'fontWeight': 'bold'}),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'Reset'},
                {'label': 'Water Rescue', 'value': 'Water Rescue'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '25px'}
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),

        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        selected_columns=[],

        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '180px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        },
        style_header={
            'fontWeight': 'bold',
            'backgroundColor': 'lightgrey'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'Water Rescue':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'Mountain or Wilderness Rescue':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'Disaster or Individual Tracking':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return []

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Breed Distribution for Selected Rescue Type'
            )
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return []

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[
                        dff.iloc[row]['location_lat'],
                        dff.iloc[row]['location_long']
                    ],
                    children=[
                        dl.Tooltip(str(dff.iloc[row]['breed'])),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(dff.iloc[row]['name']))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app
app.run_server()

Dash app running on https://carrottrident-validhelena-3000.codio.io/proxy/8050/
